# 05. LifeMate AI Agent — Sinh Bài Nộp Nhanh (Chỉ mất ~20 phút)

> **Mục tiêu:** Nạp bộ não LoRA đã huấn luyện thành công từ Hugging Face (`platypus123/lifemate-qwen3-4b-lora`) để suy luận toàn bộ 634 mẫu Public Test.
> **Thời gian chạy:** Chỉ khoảng **~20 – 25 phút** (HOÀN TOÀN KHÔNG CẦN TRAIN LẠI!).
> **Đầu ra:** File `submission_public_test.jsonl` chuẩn hợp lệ 7 tầng để nộp thẳng lên Leaderboard Zalo AI Challenge.

## 1. Kiểm tra Card Đồ Họa (GPU Status)

In [ ]:
!nvidia-smi

## 2. Cài đặt các thư viện AI & Gỡ bỏ torchao xung đột

In [ ]:
!pip uninstall -y torchao
!pip install -q "transformers>=4.45" accelerate peft bitsandbytes datasets sentencepiece huggingface_hub

## 3. Clone Mã Nguồn vào `/kaggle/working/LifeMateAgent`

In [ ]:
import os
from pathlib import Path

WORKSPACE = Path("/kaggle/working/LifeMateAgent")
if not WORKSPACE.exists():
    !git clone https://github.com/Platypus27-coder/zalo-ai-challenge.git {WORKSPACE}

%cd {WORKSPACE}
!git checkout main
!git pull origin main

## 4. Chuẩn bị Dữ Liệu Public Test
Hệ thống tự động liên kết thư mục `public_test` từ `/kaggle/input/`.

In [ ]:
import os, shutil
from pathlib import Path

DATA_DIR = WORKSPACE / "train+public_test"
DATA_DIR.mkdir(parents=True, exist_ok=True)

if Path("/kaggle/input").exists():
    found_test = list(Path("/kaggle/input").rglob("public_test.jsonl"))
    found_train = list(Path("/kaggle/input").rglob("train.jsonl"))

    if found_test and not (DATA_DIR / "public_test" / "public_test.jsonl").exists():
        test_src = found_test[0].parent
        target_test = DATA_DIR / "public_test"
        try:
            os.symlink(str(test_src), str(target_test))
            print(f"🔗 Đã liên kết public_test từ: {test_src}")
        except Exception:
            !cp -r "{test_src}" "{target_test}"

    if found_train and not (DATA_DIR / "train" / "train.jsonl").exists():
        train_src = found_train[0].parent
        target_train = DATA_DIR / "train"
        try:
            os.symlink(str(train_src), str(target_train))
        except Exception:
            !cp -r "{train_src}" "{target_train}"

if (DATA_DIR / "public_test" / "public_test.jsonl").exists():
    print("✅ Dữ liệu public_test (634 mẫu) đã sẵn sàng 100%!")
else:
    print("⚠️ Chưa thấy public_test.jsonl! Kiểm tra lại mục + Add Input.")

## 5. Sinh Bài Nộp Cho 634 Mẫu Public Test (Thời gian: ~20 – 25 phút)
Nạp trực tiếp Não bộ LoRA từ Hugging Face `platypus123/lifemate-qwen3-4b-lora` để suy luận.

In [ ]:
import os
from pathlib import Path

SUBMISSION_OUT = "/kaggle/working/submission_public_test.jsonl"

# ĐIỀN TOKEN HUGGING FACE CỦA BẠN (BẮT BUỘC ĐỂ NẠP ADAPTER PRIVATE):
HF_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxx"
ADAPTER_PATH = "platypus123/lifemate-qwen3-4b-lora"

# Nếu còn file adapter trong /kaggle/working thì dùng trực tiếp
local_cand = Path("/kaggle/working/qwen3_4b_lora")
if (local_cand / "adapter_model.safetensors").exists():
    ADAPTER_PATH = str(local_cand)
    print(f"⚡ Sử dụng adapter cục bộ: {ADAPTER_PATH}")
else:
    print(f"🌐 Nạp adapter trực tiếp từ Hugging Face: {ADAPTER_PATH}")

token_arg = f"--hf-token {HF_TOKEN}" if HF_TOKEN and not HF_TOKEN.startswith("hf_xxx") else ""

!CUDA_VISIBLE_DEVICES=0 python scripts/generate_submission.py \
    --split public_test \
    --asr-mode precomputed \
    --agent-mode transformers \
    --model-name Qwen/Qwen3-4B-Instruct-2507 \
    --adapter-path {ADAPTER_PATH} \
    {token_arg} \
    --output {SUBMISSION_OUT} \
    --max-steps 5 \
    --no-resume \
    --verbose


## 6. Kiểm tra Hợp Chuẩn 7 Lớp & Tải File Nộp Bài
Kiểm tra file nộp bài qua linter chính thức và hiện link tải về máy ngay lập tức.

In [ ]:
!python scripts/lint_submission.py {SUBMISSION_OUT}

from IPython.display import FileLink
print("\n" + "=" * 60)
print("🎉 CHÚC MỪNG BẠN! FILE NỘP BÀI ĐÃ SẴN SÀNG!")
print("👉 BẤM VÀO ĐƯỜNG LINK DƯỚI ĐÂY ĐỂ TẢI FILE VỀ MÁY TÍNH:")
display(FileLink(SUBMISSION_OUT))
print("=" * 60)